# Field extraction: LoRA fine-tune on a free Colab GPU

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` with LoRA (supervised fine-tuning) to read the fields of Greek
receipts from their OCR text, then measures it beside the prompted model on the golden test set.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The whole notebook takes about 15 minutes.

All the logic lives in the repository (`extraction/finetune.py`, `extraction/evaluate.py`); this notebook
only runs it, so what runs here is the same code that is tested and linted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. The code and the dataset

The dataset is committed in the repository (`data/field-extraction/v1`), split by shop. Each run starts
from a fresh copy, so running the notebook again picks up the latest code.

A shell command (`!`) that fails does not stop the notebook, so each step checks its own result and stops
with a clear message.

In [ ]:
import importlib.util

REPO = "https://github.com/edisban/Tameio.git"
BRANCH = "main"
%cd /content
!rm -rf Tameio && git clone --depth 1 --branch {BRANCH} {REPO}
%cd /content/Tameio/ml/document-ai
# Colab already has PyTorch with CUDA; this adds transformers, peft, trl and the project itself
!pip install -q -e ".[llm]"

# Python remembers what was installed before; look again
importlib.invalidate_caches()
assert importlib.util.find_spec("trl"), "The install failed: read the error above."

## 2. Train the LoRA adapter

Two epochs over the 433 training receipts. The loss counts only the JSON answer, not the receipt text.
The validation loss is printed after each epoch: it should fall, then flatten.

In [ ]:
from pathlib import Path

!python -m extraction.finetune --out models/field-extraction-lora

assert Path("models/field-extraction-lora/adapter_config.json").exists(), "Training failed: read the error above."

## 3. Measure both LLM variants on the golden test set

Writes `results/field-extraction/metrics.json` and `comparison.md`. The baseline row comes from the
committed file; the two LLM rows are measured here, on the same GPU, so their latencies compare.

In [ ]:
import json

!python -m extraction.evaluate --variants prompted finetuned --adapter models/field-extraction-lora

measured = json.loads(Path("results/field-extraction/metrics.json").read_text(encoding="utf-8"))["variants"]
assert "finetuned" in measured, "Measuring failed: read the error above."
!cat results/field-extraction/comparison.md

## 4. Bring the results back

Downloads one zip with the adapter and the results. Unzip it in `ml/document-ai` and commit the files in
`results/field-extraction/`; the adapter in `models/` is git-ignored.

In [ ]:
!zip -qr field-extraction-results.zip models/field-extraction-lora results/field-extraction -x "*/checkpoints/*"
from google.colab import files

files.download("field-extraction-results.zip")